<img src="../../data/images/hiaoos_learning_moored.png" width="300" align="right">

# Postprocessing and QC

- Clock corrections
- Drift corrections
    - Simple case (pressure drift)
    - Slightly more complex case (conductivity/salinity drift) 
- Chop deployment and recovery
- Range checks
- Outlier removal

In [2]:
from kval.data import moored
import xarray as xr

___

## Load the time series

In [3]:
# Specify where we want to load the file from
src_path = '../../data/moored_CTD_test_data/intermediate_data/'

# Specify a file name
src_name = 'AT800_21_22_CONCERTO_60595_99m_from_raw.nc'

In [4]:
ds = xr.open_dataset(src_path + src_name, decode_cf=False)

We will also make a copy of the dataset before applying edits so we can compare:

In [5]:
ds0 = ds.copy(deep = True)

`(deep=True)` makes sure that we are making a copy that will not be affected by any changes we make to `ds`.

___

## Adjust for a known clock drift

The internal clock of the instrument can drift over time while deployed. With modern RBR instruments this rarely amounts to more than some minutes of drift over years of deployment - but we should compensate for the drift in any event.

Once you know the difference between the instrument clock and true time on recovery, we can adjust the time stamp of the instrument to compensate, assuming that time drift occurs linearly over the deployment period.

Below, we imagine that we have observed that the instrument clock *leads* true time by 2 minutes and 34 seconds.

> *If the instrument clock* lags *true time, use negative values for `seconds`, `minutes` etc).*

In [6]:
ds = moored.adjust_time_for_drift(ds, minutes=2, seconds=34)

The updated `ds` should now be corrected so that the instrument clock is in line with true time on recovery.

___

## Remove deployment/deck time

TBW

___

## Adjust for a known offset or drift in measured values

Instrument sensors drift over time. Typically, temperature sensors are quite stable, but pressure sensors and especially conductivity sensors more often experience drift during long deployments. In such cases, it may be necessary to compensate for the drift. 

How to do this compensation is somewhat subjective and often depends on whether other, independent measurements are available to calibrate the instrument - and on your expertise on the study site and sampling configuration.     

>**Note** It is good practice to make any corrections you think will improve the data -  while indicating very explicitly any corrections you have applied in the dataset metadata so a user of the data is aware of what you have done and can make their own judgements. 


In many cases, we have laboratory calibrations of the instrument available both before and after deployment. Often, however, drift in conductivity is due not to the instrument electronics, but external factors like buildup of biological matter within the conductivity cell.

### A simple case: applying a simple offset *(X)*

Let's say we that there is a known, constant offset in pressure. We can use the following function to  

In [33]:
# TBW - wrap this function in moored..

### A simple case: adjusting for a simple pressure drift

Let's continute with pressure as an exaple. Assume that we know that a sensor was measuring correctly on deployment, but there is a known difference `Δp` between instrument and true pressure on recovery. For example, this could be because:
- We have compared instrument pressure around recovery with independent measurements from a different, well-calibrated instrument, or:
- We observe a drift in pressure while we know that the depth was constant.


The following adjusts the pressure to compensate for the offset on recovery - applying a correction that linearly drifts from 0 on deployment `Δp` on recovery.


In [27]:
deltaP = 0.1
ds = moored.linear_drift_offset(ds, 'PRES', end_val = deltaP)

The same can be done for any other variable besides `PRES`. 

### Compensating for known salinity drift

Salinity is calculated from temperature, pressure and conductivity. If we have drift in salinity this is typically due to a drift in conductivity.

> **ØF - Refer to RBR documentation on whether corrections shoudl be applied as a factor**

In some cases, we may deem it sufficient to apply a fixed offset of offset drift like we did for pressure above. However,

- It is the conductivity that drifts, we should ideally make corrections to conductivity and then recalculate salinity.
- Conductivity drift corrections should typically be applied as a *factor*, not an offset. I.e. instead of *adding* an evolving offset, we should apply a linearly evolving factor.

For example, lets say that we know initial conductivity (measurements on deployment) were correct, but we ob serve an offset in salinity `ΔS` on recovery. Ideally, we shoudl then:

1. Find the conductivity offset `ΔC` corresponding to the salinity offset `ΔS` at the temperature and pressure `ΔS` was measured.
2. Find the *factor* `αC` thath we need to multiply our deployment conductivity with in order to correct it to true conductivity.